In [1]:
import gwsus as sus
import gwsus.utils as utils
import gwsus.materials as materials
import gwsus.shapes3d as shapes
from gwsus import info
from gwsus import anime

<IPython.core.display.Javascript object>

In [2]:
import sympy as sp
import mpmath as mpm # for multiple precision numerical mathematics

In [3]:
def is_symmetric(A):
    return all(
        A[i, j] == A[j, i]
        for i in range(A.rows)
        for j in range(i)
    )


def find_zero_rows(A):
    """Return the indices of exactly zero rows in matrix A."""
    return [
        i for i in range(A.rows)
        if all(A[i, j] == 0 for j in range(A.cols))
    ]


def find_zero_cols(A):
    """Return the indices of exactly zero columns in matrix A."""
    return [
        j for j in range(A.cols)
        if all(A[i, j] == 0 for i in range(A.rows))
    ]

def smallest_nonzero_entries(A, n=10, dps=80):
    """Return the n smallest nonzero entries by numerical magnitude."""
    import mpmath as mpm

    entries = []

    with mpm.workdps(dps):
        for i in range(A.rows):
            for j in range(A.cols):
                value = A[i, j]

                if value != 0:
                    magnitude = abs(
                        mpm.mpf(str(value.n(dps)))
                    )
                    entries.append((magnitude, i, j, value))

        entries.sort(key=lambda item: item[0])

    return entries[:n]

# Specification of bodies and wires

In [4]:
# Ground specification
pos_GND = { 'for_F1' : utils.center( 0 ) }
GND_data = sus.GroundData( pos = pos_GND )

In [5]:
# GAS filter specification

# mass = 100.21
# f0 = 0.250
# saturation = -1e-3
# loss_angle = 1e-4
# moi = utils.matrix_3x3_diagonal( 3.670091 , 3.662444 , 6.095939 )
# pos_F1 = { 'top':utils.center( 1e-3 ) , 'bottom' : utils.center( -1e-3 ) }


mass = 91.31        # kg
f0 = 0.250          # Hz
saturation = -1e-3  # dimensionless
loss_angle = 1e-4   # dimensionless

moi = utils.matrix_3x3_diagonal( 3.670091 , 3.662444 , 6.095939 ) # kg⋅m²


# moi = utils.matrix_3x3( [ [ 3.8399, 0.61  , 0      ],
#                           [ 0.61  , 3.8399, 0      ],
#                           [ 0     , 0     , 6.4676 ] ] ) # kg⋅m²

# pos_F1 = { 'for_GND':utils.center( -13.5e-3 ),
#            'for_CB' : utils.center( -24.6e-3 ) } # args in m

pos_F1 = { 'for_GND':utils.center( 1e-3 ),
           'for_CB' : utils.center( -1e-3 ) } # args in m

F1_data = sus.GASFilterData( mass = mass,
                                    moi = moi,
                                    pos = pos_F1,
                                    resonance_frequency = f0,
                                    saturation = saturation,
                                    loss_angle = loss_angle,
                                    shape = shapes.GASFilterShape ) 

In [6]:
type( F1_data.loss_angle )

symengine.lib.symengine_wrapper.Rational

In [7]:
F1_data.mass

1606342507715625/17592186044416

In [8]:
# Mass specification
mass = 2
side = utils.con( 30e-3 )
moi = utils.con( 1 / 6 ) * mass * side**2 * utils.matrix_3x3_diagonal(1,1,1)
pos_CB = { 'for_F1' : utils.center( 3e-3 ) } 
CB_data = sus.BodyData( mass = mass ,
                        moi = moi ,
                        pos = pos_CB ,
                        shape = shapes.CubeShape )

In [9]:
print( type( CB_data.moi ) )
CB_data.moi

<class 'symengine.lib.symengine_wrapper.MutableDenseMatrix'>


[7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616, 0, 0]
[0, 7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616, 0]
[0, 0, 7015207859188333098725736953443361932228438821/23384026197294446691258957323460528314494920687616]

In [10]:
# number = 1
# material = materials.maraging_steel
# diameter = 3e-3
# length = utils.con( 26.5e-3 + 1280e-3 + 26.5e-3 )
# wireF1_data = utils.WireData( number = number , material = material , diameter = diameter , length = length )

number = 1
material = materials.maraging_steel
diameter = ( 3e-3 , 4.5e-3 , 3e-3 )      # m
length = ( 26.5e-3 , 1280e-3 , 26.5e-3 ) # m

wireF1_data = sus.WireData( number = number,
                        material = material,
                        diameter = diameter,
                        length = length )

In [11]:
number = 1
material = materials.maraging_steel
diameter = 2.5e-3 
length = 26.5e-3 + 513.4e-3 + 26.5e-3
wireCB_data = sus.WireData( number = number , material = material , diameter = diameter , length = length )

# Declare bodies and wires and connect them

In [12]:
# Declare suspension
TS = sus.Suspension('TS')

In [13]:
# Declare body instances
GND = sus.Ground( 'GND' , data = GND_data )
F1 = sus.GASFilter( 'F1' , data = F1_data )
CB  = sus.RigidBody( 'CB' , data = CB_data )

F1: the attachment point at the lower end of the internal spring is called 'for_CB'.


In [14]:
# Declare link instances
wire_F1 = sus.Wire( data = wireF1_data )
wire_CB = sus.Wire( data = wireCB_data )

In [15]:
# Connect ground and cube with spring
TS.connect( GND , 'for_F1' , F1 , 'for_GND' , wire_F1 )
TS.connect( F1 , 'for_CB' , CB , 'for_F1' , wire_CB )
GND.show()

GND
└── F1
    └── CB


In [16]:
# print( F1.spring_body )

In [17]:
TS.update()

F1: beta, m1, m2 = -0.001 , 0.1 , 0.0360453484494832


In [18]:
F1.link_internal

In [19]:
info.show_info( TS , 'load')

╭────────┬─────────────┬─────────────┬────────────────┬───────────────╮
│ Name   │   Mass [kg] │   Load [kg] │      Number of │   Single link │
│        │             │             │   links in set │     load [kg] │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ GND    │             │       93.31 │                │               │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ F1     │       91.31 │        2    │              1 │         93.31 │
├────────┼─────────────┼─────────────┼────────────────┼───────────────┤
│ CB     │        2    │        0    │              1 │          2    │
╰────────┴─────────────┴─────────────┴────────────────┴───────────────╯


In [20]:
info.show_info( TS , 'links')

╭────────┬──────────────┬────────────────┬───────────────┬─────────────┬─────────────╮
│ Name   │     Link set │      Number of │   Single link │       Axial │   Torsional │
│        │   total load │   links in set │     load [kg] │   stiffness │   stiffness │
│        │         [kg] │                │               │       [N/m] │    [Nm/rad] │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ GND    │              │                │               │             │             │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ F1     │        93.31 │              1 │         93.31 │ 2.21643e+06 │      5353.1 │
├────────┼──────────────┼────────────────┼───────────────┼─────────────┼─────────────┤
│ CB     │         2    │              1 │          2    │ 1.68998e+06 │     24408   │
╰────────┴──────────────┴────────────────┴───────────────┴─────────────┴─────────────╯


In [21]:
F1.link.k_torsion.n(50)

5353.10360885206

In [22]:
CB.link.k_torsion.n(50)

24408.0426739088

In [23]:
print( TS.float_coords_list )

None


In [24]:
print( TS.equi_coords_guess_dict )

None


In [25]:
TS.calculate_equilibrium_positions(decimal_places= 150)

Let's calculate the equilibrium positions...
Tolerance: 1.0e-145
Guessing the values of the equilibrium coordinates... Calculating the total potential energy...
	Calculating the total wire axial elastic potential energy... Done.
	I'm reasonably assuming that the link elastic torsional potential energies are zero at equilibrium.
Calculating symbolic gradient...
lonCB
term_num: 0

lonF1
term_num: 0

pitchCB
term_num: 0

pitchF1
term_num: 0

Nan detected
qF1
term_num: 0

rollCB
term_num: 0

rollF1
term_num: 0

traCB
term_num: 0

traF1
term_num: 0

verCB
term_num: 0

verF1
term_num: 0

Done.
Calculating symbolic gradient...
Converting gradient from Symengine to Mpmath... 0.0656 seconds.
Minimizing the total potential energy... 

ZeroDivisionError: 

In [ ]:
info.print_equilibrium_positions(TS)

In [ ]:
TS.equi_coords_dict

In [ ]:
print( F1.link.Ep_dict['torsion']['energy'].n(20) )
print( CB.link.Ep_dict['torsion']['energy'].n(20) )

In [ ]:
print( F1.link.Ep_dict['axial']['energy'].n(20) )
print()
print( CB.link.Ep_dict['axial']['energy'].n(20) )

In [ ]:
print( TS.float_coords_list )

In [ ]:
TS.calculate_mass_matrix()

In [ ]:
print( TS.float_coords_list )

In [ ]:
TS.calculate_stiffness_matrix( with_damping = False )

In [ ]:
print( TS.P.shape )
print( TS.M_matrix.shape )

In [ ]:
print( is_symmetric( TS.M_matrix ) )
print( is_symmetric( TS.P ) )

In [ ]:
zero_rows = find_zero_rows(TS.P)
zero_cols = find_zero_cols(TS.P)

print("Zero rows:", zero_rows)
print("Zero columns:", zero_cols)

In [ ]:
for magnitude, i, j, value in smallest_nonzero_entries(TS.P):
    print(f"K[{i}, {j}] = {value}")
    print(f"    |K[{i}, {j}]| ≈ {magnitude}")

In [ ]:
TS.calculate_eigenvectors( TS.M_matrix )

In [ ]:
# TS.calculate_eigenvectors( TS.P )

In [ ]:
# TS.calculate_eigenvectors()
# TS.calculate_eigenvectors(decimal_places = 80)

In [ ]:
for prec in (50, 80, 100):
    TS.calculate_eigenvectors(TS.P, decimal_places = prec)
    print('\n')
    # for key, value in TS.eigen_dict.items():
    #     print( key, mpm.nstr( value['frequency'],6 ) )
    print('\n')
    print('\n')
    

In [ ]:
info.print_eigenfreq(TS)

In [ ]:
n = 1
info.print_eigen( TS , n )

In [ ]:
TS.coords_list

In [ ]:
with mpm.workdps(100):
    for i in [5, 12]:
        row = []
        for j in [5, 12]:
            value = TS.P[i, j]
            value_mp = mpm.mpf(str(value.n(mpm.mp.dps)))
            row.append(mpm.nstr(value_mp, 30))
        print(row)

In [ ]:
for prec in (50, 80, 100):
    TS.calculate_eigenvectors(decimal_places = prec)
    print('\n')
    for key, value in TS.eigen_dict.items():
        print( key, mpm.nstr( value['frequency'],6 ) )
    print('\n')
    print('\n')
    

In [ ]:
info.print_eigenfreq( TS )

In [ ]:
n = 2
info.print_eigen( TS , n )

In [ ]:
assert False, "Stopped execution here"

In [ ]:
anime.prepare_anime( TS )

In [ ]:
n = 7
# info.print_eigen( TS , n )
# anime.show_anime( TS , n )

In [ ]:
print( F1.spring_load )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
TS.calculate_stiffness_matrix( with_damping = True )

In [ ]:
TS.prepare_transfer_functions()

In [ ]:
frequency = np.logspace(-2,4,1000)

In [ ]:
TS.body_coords_list

In [ ]:
in_coord = GND.ver
out_coord = CB.ver
tf = TS.transfer_function( in_coord , out_coord )

In [ ]:
plt.loglog( frequency , np.abs( tf( 2*np.pi*frequency ) ) )
plt.axvline(x=0.25, color='r', linestyle='--', linewidth=2)
plt.axhline(y=1e-3, color='r', linestyle='--')

In [ ]:
7771980404306739/274877906944

In [ ]:
import symengine

# Create a 0x1 dense matrix (vector)
empty_vec = symengine.Matrix(1, 0, []).T

# Convert to list
result = empty_vec.T.tolist()

print(result)  # Output: []
print(type(result))  # Output: <class 'list'>


In [ ]:
import symengine
from symengine import DenseMatrix

# Create a 0x1 empty column vector
v = DenseMatrix(1, 0, []).T 

# Transpose it to a 1x0 row vector
v_T = v.T

# Convert to list: results in a nested empty list [[]]
result = v_T.tolist()

print(f"Original: {v.tolist()}") # Output: []
print(f"Transposed: {result}")   # Output: [[]]
print(f"Shape: {v_T.shape}")      # Output: (1, 0)


In [ ]:
import symengine as se

In [ ]:
isinstance( utils.con( 5 , 5 , 3 ) , se.Number )

In [ ]:
utils.con( 30 , 43 , 6 )

In [ ]:
F1.omega

In [ ]:
F1.omega[2]

In [ ]:
my_keys = GND.gnd_coords
my_values = [ utils.con(0) ] * len( my_keys )
dict( zip( my_keys , my_values ) ) 

In [ ]:
aa = [1.23 , 6.43]

In [ ]:
bb = [aa] if not isinstance( aa , ( list , tuple ) ) else aa

In [ ]:
bb

In [ ]:
utils.con( 1.23 , 6.43 )

In [ ]:
utils.con( 1.23 )